In [10]:
%pip install -qU \
    langchain \
    langchain-core \
    langchain-groq \
    langchain-huggingface \
    langchain-pinecone \
    pinecone \
    sentence-transformers \
    pymupdf \
    groq \
    pandas \
    tabulate \
    pillow \
    python-dotenv

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crawl4ai 0.8.9 requires chardet>=5.2.0, which is not installed.
crawl4ai 0.8.9 requires pyOpenSSL>=25.3.0, but you have pyopenssl 24.3.0 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:

%pip uninstall -y pillow
%pip install --no-cache-dir --force-reinstall pillow

Found existing installation: pillow 12.3.0
Uninstalling pillow-12.3.0:
  Successfully uninstalled pillow-12.3.0
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 20.9 MB/s eta 0:00:00a 0:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
crawl4ai 0.8.9 requires chardet>=5.2.0, which is not installed.
crawl4ai 0.8.9 requires pyOpenSSL>=25.3.0, but you have pyopenssl 24.3.0 which is incompatible.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import io
import time
import base64
from pathlib import Path
from getpass import getpass

import fitz
import pandas as pd

from PIL import Image
from IPython.display import display

from groq import Groq

from pinecone import Pinecone, ServerlessSpec

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_pinecone import PineconeVectorStore

/home/kompallygiridhar/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
PDF_PATH = Path("/home/kompallygiridhar/Desktop/Learning/RAG_Concepts/MultiModal_RAG_Implementation/NovaCore_Multimodal_Company_Report_2026.pdf")

if not PDF_PATH.exists():
    fallback = Path("/mnt/data/")  # For Colab
    if fallback.exists():
        PDF_PATH = fallback


assert PDF_PATH.exists(), f"PDF file not found at {PDF_PATH}. Please upload the file to this path."

In [4]:

TEXT_MODEL = "openai/gpt-oss-20b"
VISION_MODEL = "qwen/qwen3.8-27b"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

PINECONE_INDEX_NAME = "multimodal-rag"
PINECONE_NAMESPACE = "multimodal-rag-namespace"

IMAGE_DIR = Path("multimodal_extracted_images")
IMAGE_DIR.mkdir(exist_ok=True)

print("PDF:", PDF_PATH)
print("Pinecone index:", PINECONE_INDEX_NAME)
print("Pinecone namespace:", PINECONE_NAMESPACE)

PDF: /home/kompallygiridhar/Desktop/Learning/RAG_Concepts/MultiModal_RAG_Implementation/NovaCore_Multimodal_Company_Report_2026.pdf
Pinecone index: multimodal-rag
Pinecone namespace: multimodal-rag-namespace


In [7]:
from dotenv import find_dotenv, load_dotenv


env_file = find_dotenv(usecwd=True)
if not env_file:
    raise FileNotFoundError("Could not find a .env file from the notebook's working directory.")

load_dotenv(env_file)

required_keys = ("GROQ_API_KEY", "PINECONE_API_KEY")
missing_keys = [key for key in required_keys if not os.getenv(key)]
if missing_keys:
    raise RuntimeError(f"Missing environment variables in .env: {', '.join(missing_keys)}")

print("Environment variables loaded successfully.")

groq_client = Groq(api_key=os.getenv("GROQ_API_KEY"))

text_llm = ChatGroq(model=TEXT_MODEL, api_key=os.getenv("GROQ_API_KEY"),temperature=0.0)

Environment variables loaded successfully.


In [ ]:
def 

In [14]:
def image_to_base64(image_path: Path) -> str:
    """Convert an image file to a base64-encoded string."""
    image = Path(image_path)

    with Image.open(image) as img:
        img = img.convert("RGB")  # Ensure image is in RGB format
        img.thumbnail((1600, 1600))  # Resize to a maximum of 1600x1600 pixels

        buffered = io.BytesIO()
        img.save(buffered, format="JPEG",quality=85)  # Save as JPEG with quality 85

    encoded = base64.b64encode(buffered.getvalue()).decode("utf-8")
    return f"data:image/jpeg;base64,{encoded}"

In [8]:
def summarize_visual(image_path, page_number):
    image_data = image_to_base64(image_path)

    prompt = f"""
This visual was extracted from page {page_number} of the NovaCore FY2026 company report.

Describe the useful business information visible in the visual.

If it is a chart or graph:
- mention important values
- mention highest/lowest values
- mention the main trend

If it is a diagram:
- identify important components
- explain the flow or relationships

If it is a normal business image:
- describe the useful factual information

Keep the description concise and factual.
"""

    response = groq_client.chat.completions.create(
        model=VISION_MODEL,
        messages=[
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": prompt},
                    {
                        "type": "image_url",
                        "image_url": {"url": image_data},
                    },
                ],
            }
        ],
        temperature=0,
        max_completion_tokens=500,
    )

    return response.choices[0].message.content.strip()

In [9]:
def extract_multimodal_documents(pdf_path):
    pdf = fitz.open(str(pdf_path))

    documents = []
    extracted_xrefs = set()

    for page_index in range(len(pdf)):
        page = pdf[page_index]
        page_number = page_index + 1

        print(f"Processing page {page_number}...")

        # -------------------------------------------------
        # 1. TEXT
        # -------------------------------------------------
        text = page.get_text("text").strip()

        if text:
            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "page": page_number,
                        "modality": "text",
                        "source": pdf_path.name,
                    },
                )
            )

        # -------------------------------------------------
        # 2. TABLES
        # -------------------------------------------------
        try:
            tables = page.find_tables().tables

            for table_number, table in enumerate(tables, start=1):
                df = table.to_pandas()

                if not df.empty:
                    table_text = df.to_markdown(index=False)

                    documents.append(
                        Document(
                            page_content=table_text,
                            metadata={
                                "page": page_number,
                                "modality": "table",
                                "table_number": table_number,
                                "source": pdf_path.name,
                            },
                        )
                    )

        except Exception as error:
            print("Table extraction warning:", error)

        # -------------------------------------------------
        # 3. IMAGES / CHARTS / DIAGRAMS
        # -------------------------------------------------
        for image_number, image_info in enumerate(
            page.get_images(full=True),
            start=1,
        ):
            xref = image_info[0]

            # Do not process an identical embedded image twice.
            if xref in extracted_xrefs:
                continue

            extracted_xrefs.add(xref)

            image_data = pdf.extract_image(xref)
            image_bytes = image_data["image"]
            image_extension = image_data["ext"]

            image_path = (
                IMAGE_DIR
                / f"page_{page_number}_image_{image_number}.{image_extension}"
            )

            image_path.write_bytes(image_bytes)

            try:
                summary = summarize_visual(
                    image_path=image_path,
                    page_number=page_number,
                )

                documents.append(
                    Document(
                        page_content=summary,
                        metadata={
                            "page": page_number,
                            "modality": "visual",
                            "image_path": str(image_path),
                            "source": pdf_path.name,
                        },
                    )
                )

            except Exception as error:
                print(
                    f"Vision warning on page {page_number}:",
                    error,
                )

    pdf.close()
    return documents


documents = extract_multimodal_documents(PDF_PATH)

print()
print("Total LangChain documents:", len(documents))

Processing page 1...
Consider using the pymupdf_layout package for a greatly improved page layout analysis.
Vision warning on page 1: name 'image_to_base64' is not defined
Processing page 2...
Processing page 3...
Vision warning on page 3: name 'image_to_base64' is not defined
Processing page 4...
Vision warning on page 4: name 'image_to_base64' is not defined
Processing page 5...
Vision warning on page 5: name 'image_to_base64' is not defined
Processing page 6...
Vision warning on page 6: name 'image_to_base64' is not defined
Processing page 7...
Vision warning on page 7: name 'image_to_base64' is not defined
Processing page 8...
Vision warning on page 8: name 'image_to_base64' is not defined
Processing page 9...

Total LangChain documents: 17
